# 04 — Poster ROI Features (YOLOv8n + ResNet50 crops)

Pipeline:
1) Detect salient boxes with YOLOv8n (confidence×area ranking)
2) Take TOPK crops (or fallback to global center crop if nothing detected)
3) ResNet50 features on each crop (timm, `num_classes=0`)
4) Pool mean + max → 4096-d vector → L2-normalize → save per `tmdb_id`
Outputs per-id `.npy` and `roi_index.csv`. Param via env vars: `IMG_DIR`,
`ART_DIR`, `ROI_DIR`, `TOPK`, `BATCH_SIZE`.

In [ ]:
import os, json, joblib, numpy as np, pandas as pd
from pathlib import Path

ART_DIR   = Path(os.getenv("ART_DIR",   "~/boxoffice/artifacts")).expanduser()
DATA_DIR  = Path(os.getenv("DATA_DIR",  "~/boxoffice/data")).expanduser()
IMG_DIR   = Path(os.getenv("IMG_DIR",   "~/boxoffice/data/img")).expanduser()

TEXT_DIR  = Path(os.getenv("TEXT_DIR",  ART_DIR / "text_emb")).expanduser()
CLIP_DIR  = Path(os.getenv("CLIP_DIR",  ART_DIR / "clip_emb")).expanduser()
ROI_DIR   = Path(os.getenv("ROI_DIR",   ART_DIR / "roi_feat")).expanduser()

def load_json(path): return json.loads(Path(path).read_text(encoding="utf-8"))

# Handy helper
def _vec(path, dim):
    try:
        v = np.load(path); v = np.asarray(v, np.float32).ravel()
        if   v.size == dim: return v
        elif v.size >  dim: return v[:dim]
        else:               return np.pad(v, (0, dim - v.size))
    except Exception:
        return np.zeros(dim, np.float32)


In [ ]:
## FOR RELOAD ONLY

roi_index = pd.read_csv(ART_DIR / "roi_index.csv")
roi_meta  = load_json(ART_DIR / "roi_model_meta.json")
# vec = np.load(ROI_DIR / f"{tmdb_id}.npy")

In [1]:
import os as _os
_os.environ["TQDM_NOTEBOOK"] = "0"  # no ipywidgets
from tqdm import tqdm                
try:
    tqdm.monitor_interval = 0        
except Exception:
    pass
print("tqdm patched: text mode, monitor thread disabled")

tqdm patched: text mode, monitor thread disabled


In [2]:
# ---- params & setup ----
import os, math, gc
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image

IMG_DIR   = Path(os.getenv("IMG_DIR", os.path.expanduser("~/boxoffice/data/img")))
ART_DIR   = Path(os.getenv("ART_DIR", os.path.expanduser("~/boxoffice/artifacts")))
ROI_DIR   = Path(os.getenv("ROI_DIR", str(ART_DIR / "roi_feat")))
TOPK      = int(os.getenv("TOPK", "5"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "16"))
ROI_DIR.mkdir(parents=True, exist_ok=True)

print("IMG_DIR:", IMG_DIR)
print("ROI_DIR:", ROI_DIR)
print("TOPK:", TOPK, "BATCH_SIZE:", BATCH_SIZE)

IMG_DIR: C:\Users\Vaishob\boxoffice\data\img
ROI_DIR: C:\Users\Vaishob\boxoffice\artifacts\roi_feat
TOPK: 5 BATCH_SIZE: 16


In [3]:
# ---- models ----
import torch
from ultralytics import YOLO
import timm
import torchvision.transforms as T

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Detector (downloads yolov8n.pt on first run)
det_model = YOLO("yolov8n.pt")
det_model.to(device)

# CNN trunk for crops (feature extractor)
cnn = timm.create_model("resnet50", pretrained=True, num_classes=0)
cnn.eval().to(device)

# Preprocess for CNN
cnn_tf = T.Compose([
    T.Resize(256, interpolation=T.InterpolationMode.BILINEAR),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Vaishob\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device: cpu


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

In [4]:
# ---- helpers ----
def l2_normalize(x, eps=1e-9):
    import numpy as np
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return (x / np.maximum(n, eps)).astype(np.float32)

def load_image(path):
    return Image.open(path).convert("RGB")

def crop_boxes(image_pil, boxes_xyxy, topk):
    W, H = image_pil.size
    boxes = boxes_xyxy[:topk]
    crops = []
    for (x1,y1,x2,y2) in boxes:
        x1 = max(0, int(x1)); y1 = max(0, int(y1))
        x2 = min(W, int(x2)); y2 = min(H, int(y2))
        if x2 > x1 and y2 > y1:
            crops.append(image_pil.crop((x1,y1,x2,y2)))
    return crops

import torch
@torch.no_grad()
def featurize_crops(pil_list, BATCH_SIZE=16):
    if not pil_list:
        return None
    import numpy as np, torch
    ten = torch.stack([cnn_tf(p) for p in pil_list]).to(device)
    feats = []
    for i in range(0, ten.shape[0], BATCH_SIZE):
        batch = ten[i:i+BATCH_SIZE]
        f = cnn(batch).float().cpu().numpy()
        feats.append(f)
    X = np.vstack(feats)  # (n_crops, d=2048 for resnet50)
    return X

In [5]:
# ---- discover images ----
ids = []
for p in IMG_DIR.glob("*.jpg"):
    try:
        ids.append(int(p.stem))
    except:
        pass
ids = sorted(set(ids))
print("Found posters:", len(ids))

Found posters: 4177


In [6]:
# ---- main loop ----
index_rows = []
for tmdb_id in tqdm(ids):
    path = IMG_DIR / f"{tmdb_id}.jpg"
    if not path.exists():
        continue
    try:
        im = load_image(path)
        # run detector (device=0 for cuda, None for cpu)
        res = det_model.predict(source=str(path), device=0 if device=='cuda' else None, verbose=False)[0]
        boxes, scores = [], []
        W, H = im.size
        if res and hasattr(res,'boxes') and res.boxes is not None:
            xyxy = res.boxes.xyxy.cpu().numpy() if hasattr(res.boxes,'xyxy') else None
            conf = res.boxes.conf.cpu().numpy() if hasattr(res.boxes,'conf') else None
            if xyxy is not None and conf is not None:
                for b,c in zip(xyxy, conf):
                    x1,y1,x2,y2 = b
                    area = max(0,(x2-x1))*max(0,(y2-y1))
                    boxes.append((x1,y1,x2,y2))
                    scores.append(float(c)*float(area))

        # rank by (conf * area)
        if boxes:
            import numpy as np
            order = np.argsort(scores)[::-1]
            boxes_sorted = [boxes[i] for i in order]
            crops = crop_boxes(im, boxes_sorted, TOPK)
        else:
            crops = []

        # fallback if no detections
        if not crops:
            crops = [im]

        X = featurize_crops(crops, BATCH_SIZE=BATCH_SIZE)  # (n, 2048)
        if X is None:
            vec = np.zeros((2048,), dtype=np.float32)
        else:
            mean = X.mean(axis=0)
            mx   = X.max(axis=0)
            vec = np.concatenate([mean, mx], axis=0).astype(np.float32)  # 4096
        vec = l2_normalize(vec.reshape(1,-1)).squeeze(0)

        np.save(ROI_DIR / f"{tmdb_id}.npy", vec)
        index_rows.append({"tmdb_id": int(tmdb_id), "dim": int(vec.shape[0]), "crops_used": int(len(crops))})
        gc.collect()

    except Exception as e:
        print("Skip", tmdb_id, "error:", e)

pd.DataFrame(index_rows).to_csv(ROI_DIR / "roi_index.csv", index=False)
print("Saved ROI features:", len(index_rows))

100%|██████████████████████████████████████████████████████████████████████████████| 4177/4177 [55:23<00:00,  1.26it/s]

Saved ROI features: 4177


In [8]:
# If not already written by the notebook:
from glob import glob
roi_ids = sorted(int(Path(p).stem) for p in glob(str(ROI_DIR / "*.npy")))
pd.DataFrame({"tmdb_id": roi_ids, "dim":[4096]*len(roi_ids)}).to_csv(ART_DIR / "roi_index.csv", index=False)
save_json({"detector":"YOLOv8n","backbone":"timm resnet50","pooling":"mean+max concat"}, ART_DIR / "roi_model_meta.json")